# When2Tool Interaction Round Budget Comparison
The only experimental variable is the maximum rounds per task: 6 for baseline6 versus 3 for budget3. Both conditions use the author's Default + no_reasoning configuration with the same DeepSeek model, tools, parser, and prompts. No training, deduplication, or Reason-then-Act is used.
This is a parameter modification / budget ablation permitted by the assignment, not a new algorithm. Six rounds is this project's previous baseline setting; it is not claimed to be the setting for all experiments in the original paper. Using an API and a subset does not directly reproduce the paper's full-benchmark scores.
One round consists of one model response and its processing. Rejected formatting or reasoning outputs also consume a round. No answer is automatically supplied when the budget is exhausted.
Additional analysis at no API cost: truncate and replay each baseline6 trajectory at three rounds to count tasks rescued by rounds 4–6, avoiding attributing random differences between independent API runs entirely to the round budget.


In [ ]:
from pathlib import Path
import subprocess,sys,os,json,time
ROOT=Path('/content/when2tool-round-budget')
if not ROOT.exists():
    subprocess.run(['git','clone','https://github.com/Trustworthy-ML-Lab/when2tool.git',str(ROOT)],check=True)
subprocess.run(['git','checkout','ef0cf1ea72eb8acdfd3fb1251c4b6a9a0131a23e'],cwd=ROOT,check=True)
os.chdir(ROOT)
subprocess.run([sys.executable,'data_generators/gen_calculator_20.py'],check=True)


In [ ]:
source='"""DeepSeek adapter for the author\'s baseline state machine (no torch needed)."""\nimport argparse\nimport json\nimport os\nfrom pathlib import Path\nimport random\nimport re\nimport sys\nimport time\nimport urllib.request\nsys.path.insert(0,str(Path(__file__).parent/\'src\'))\nimport utils\nROOT=Path(__file__).resolve().parent\n\n\ndef parse(text):\n    match=re.search(r\'<tool_call>(.*?)</tool_call>\',text,re.S)\n    if match:\n        try:\n            call=json.loads(match.group(1))\n            if isinstance(call,dict) and isinstance(call.get(\'arguments\'),dict) and isinstance(call.get(\'name\'),str):\n                return dict(type=\'tool\',tool_name=call[\'name\'],arguments=call[\'arguments\'],raw_text=text)\n        except (ValueError,TypeError):pass\n    return dict(type=\'content\',content=text,raw_text=text)\n\n\ndef run_episode(task,generate,rounds=6):\n    st=utils.init_state(task,utils.SYSTEM_PROMPT_XML,record_mode=\'full\',prompt_mode=\'current\',require_reasoning=False,tool_format=\'xml\')\n    events=[]\n    for _ in range(rounds):\n        if st[\'done\']:break\n        out=parse(generate(st[\'messages\'],st[\'tools\']))\n        trace=dict(parsed_output=out,model_raw_output=out[\'raw_text\'])\n        utils.step_state(st,out,trace_item=trace)\n        events.append(dict(round=st[\'rounds\'],parsed=out,tool_result=trace.get(\'tool_result\'),done=st[\'done\']))\n    item=utils.finalize_state(st)\n    item.update(events=events,parse_failures=sum(\'<tool_call>\' in e[\'parsed\'][\'raw_text\'] and e[\'parsed\'][\'type\']!=\'tool\' for e in events),\n                policy_rejections=sum(str((e.get(\'tool_result\') or {}).get(\'message\',\'\')).startswith(\'Tool call rejected:\') for e in events),\n                final_correct=bool(utils.item_final_eval(item)[-1]),strict_correct=bool(st[\'done\'] and utils.item_final_eval(item)[-1]),done=st[\'done\'],budget_exhausted=not st[\'done\'],round_limit=rounds)\n    return item\n\n\ndef main():\n    p=argparse.ArgumentParser()\n    p.add_argument(\'--mock\',action=\'store_true\',help=\'Local synthetic responses, no API\')\n    p.add_argument(\'--per-difficulty\',type=int,default=1)\n    p.add_argument(\'--runs\',type=int,default=1)\n    p.add_argument(\'--max-rounds\',type=int,choices=[3,6],default=6)\n    p.add_argument(\'--output\',default=\'outputs/pilot.json\')\n    a=p.parse_args()\n    if not 1<=a.per_difficulty<=50 or a.runs<1:p.error(\'Invalid sample size\')\n    env=ROOT/\'.env\'\n    if env.exists():\n        for line in env.read_text().splitlines():\n            if line.strip() and not line.lstrip().startswith(\'#\') and \'=\' in line:\n                k,v=line.split(\'=\',1);os.environ.setdefault(k.strip(),v.strip())\n    if not a.mock and not os.environ.get(\'DEEPSEEK_API_KEY\'):\n        p.error(\'Set DEEPSEEK_API_KEY locally in .env; never paste it into chat.\')\n    tasks=[];rng=random.Random(42)\n    for difficulty in (\'easy\',\'medium\',\'hard\'):\n        path=ROOT/f\'data/task_sets/calculator_single_{difficulty}_test.json\'\n        tasks+=rng.sample(json.loads(path.read_text()),a.per_difficulty)\n    usage=dict(calls=0,attempts=0,input_tokens=0,output_tokens=0,cached_input_tokens=0)\n    api_records=[]\n    max_calls=int(os.environ.get(\'PILOT_MAX_CALLS\',\'540\'))\n    def api(messages,tools):\n        # Keep official XML protocol; make the same tool schema available as HF chat template would.\n        if usage[\'attempts\'] >= max_calls: raise RuntimeError(\'API request cap reached\')\n        usage[\'attempts\']+=1\n        msgs=[dict(m) for m in messages]\n        msgs[0][\'content\']+=\'\\nAvailable tools:\\n\'+json.dumps(tools,ensure_ascii=False)\n        body=dict(model=\'deepseek-flash\',messages=msgs,temperature=1,max_tokens=1024,thinking=dict(type=\'disabled\'))\n        req=urllib.request.Request(\'https://api.deepseek.com/chat/completions\',data=json.dumps(body).encode(),headers={\'Authorization\':\'Bearer \'+os.environ[\'DEEPSEEK_API_KEY\'],\'Content-Type\':\'application/json\'})\n        with urllib.request.urlopen(req,timeout=60) as response:d=json.load(response)\n        usage[\'calls\']+=1;usage[\'input_tokens\']+=d.get(\'usage\',{}).get(\'prompt_tokens\',0);usage[\'output_tokens\']+=d.get(\'usage\',{}).get(\'completion_tokens\',0)\n        usage[\'cached_input_tokens\']+=d.get(\'usage\',{}).get(\'prompt_cache_hit_tokens\',0)\n        api_records.append(dict(response_id=d.get(\'id\'),model=d.get(\'model\'),usage=d.get(\'usage\'),finish_reason=d[\'choices\'][0].get(\'finish_reason\')))\n        return d[\'choices\'][0][\'message\'][\'content\'] or \'\'\n    results=[];start=time.monotonic()\n    out=ROOT/a.output;out.parent.mkdir(parents=True,exist_ok=True)\n    if out.exists(): raise FileExistsError(\'Output already exists; use a new study directory\')\n    import platform\n    def save(error=None):\n        payload=dict(version=\'round-budget-1.0\',mock=a.mock,config={**vars(a),\'model\':\'deepseek-flash\',\'temperature\':1,\'max_tokens\':1024,\'thinking\':\'disabled\',\'max_rounds\':a.max_rounds,\'sample_seed\':42,\'python\':platform.python_version()},usage=usage,api_records=api_records,seconds=time.monotonic()-start,results=results,error=error)\n        tmp=out.with_suffix(\'.tmp\');tmp.write_text(json.dumps(payload,ensure_ascii=False,indent=2));tmp.replace(out)\n    for run in range(a.runs):\n        for task in tasks:\n            if a.mock:\n                # Exercise actual tool execution, then answer from tool output, not gold labels.\n                expression=task[\'instruction\'].split(\':\',1)[1].strip()\n                count=[0]\n                def generate(messages,tools):\n                    count[0]+=1\n                    if count[0]<=2:\n                        return \'<tool_call>\'+json.dumps(dict(name=\'evaluate_expression\',arguments=dict(expression=expression)))+\'</tool_call>\'\n                    for msg in reversed(messages):\n                        if \'<tool_response>\' in msg[\'content\']:\n                            result=json.loads(msg[\'content\'].split(\'<tool_response>\')[1].split(\'</tool_response>\')[0])\n                            return \'\\\\boxed{\'+str(result[\'result\'])+\'}\'\n                    raise RuntimeError(\'No tool response\')\n            else:generate=api\n            task_start=time.monotonic()\n            before=usage.copy()\n            try:\n                item=run_episode(task,generate,rounds=a.max_rounds)\n            except Exception as exc:\n                save(dict(type=type(exc).__name__,task_id=task[\'id\'],run=run))\n                raise RuntimeError(\'Run stopped; completed results saved. Inspect API settings and balance.\') from None\n            if a.max_rounds==6:\n                replay=iter(e[\'parsed\'][\'raw_text\'] for e in item[\'events\'])\n                prefix=run_episode(task,lambda *_:next(replay),rounds=3)\n                item[\'prefix3\']=dict(final_correct=prefix[\'final_correct\'],strict_correct=prefix[\'strict_correct\'],done=prefix[\'done\'],rounds=prefix[\'rounds\'])\n                item[\'late_rescue\']=bool(item[\'strict_correct\'] and not prefix[\'strict_correct\'])\n            item.update(run=run,seconds=time.monotonic()-task_start,api_usage={k:usage[k]-before[k] for k in usage});results.append(item)\n            save()\n            print(f"run {run+1}/{a.runs} task {len(results)} correct={item[\'final_correct\']} mode={item[\'reasoning_mode\']}",flush=True)\n    print(json.dumps(dict(mock=a.mock,tasks=len(results),accuracy=sum(r[\'final_correct\'] for r in results)/len(results),parse_failures=sum(r[\'parse_failures\'] for r in results),usage=usage)))\n\nif __name__==\'__main__\':main()\n'
(ROOT/'deepseek_rounds.py').write_text(source)
tests="import unittest,json\nfrom deepseek_rounds import run_episode,parse\nT=dict(id=1,difficulty='easy',instruction='Compute 2+3',environments=[dict(name='CalculatorEnv',tools=['evaluate_expression'],parameters={})],expected=dict(answer='5'))\nCALL='<tool_call>'+json.dumps(dict(name='evaluate_expression',arguments=dict(expression='2+3')))+'</tool_call>'\ndef run(seq,limit):\n it=iter(seq);return run_episode(T,lambda *_:next(it),rounds=limit)\nclass Tests(unittest.TestCase):\n def test_early_finish(self):\n  for n in (3,6):\n   x=run([CALL,r'\\boxed{5}'],n);self.assertEqual(x['rounds'],2);self.assertTrue(x['strict_correct'])\n def test_three_cap(self):\n  x=run(['continue']*3,3);self.assertEqual(x['rounds'],3);self.assertTrue(x['budget_exhausted']);self.assertFalse(x['strict_correct'])\n def test_late_recovery(self):\n  seq=['continue']*3+[CALL,r'\\boxed{5}']\n  self.assertFalse(run(seq,3)['strict_correct']);self.assertTrue(run(seq,6)['strict_correct'])\n def test_success_on_last_round(self):\n  x=run(['continue',CALL,r'\\boxed{5}'],3);self.assertTrue(x['done']);self.assertFalse(x['budget_exhausted'])\n def test_policy_unchanged(self):\n  seq=['Let me compute this carefully. '+CALL,CALL,r'\\boxed{5}']\n  for n in (3,6):self.assertEqual(run(seq,n)['policy_rejections'],1)\n def test_prefix_replay(self):\n  x=run(['continue']*3+[r'\\boxed{5}'],6)\n  it=iter(e['parsed']['raw_text'] for e in x['events']);y=run_episode(T,lambda *_:next(it),3)\n  self.assertEqual(y['rounds'],3);self.assertFalse(y['strict_correct'])\n def test_parse_invalid(self):\n  for t in ('<tool_call>[]</tool_call>','<tool_call>null</tool_call>'):self.assertEqual(parse(t)['type'],'content')\n def test_wrong_answer(self):self.assertFalse(run([r'\\boxed{6}'],3)['final_correct'])\nif __name__=='__main__':unittest.main()\n"
(ROOT/'test_rounds.py').write_text(tests)
subprocess.run([sys.executable,'-m','unittest','test_rounds','-v'],check=True)


## Free Mock Checks
Mock outputs test the implementation only; they are not experimental results. Rerunning this cell creates a new temporary directory and makes no API calls.


In [ ]:
import tempfile
with tempfile.TemporaryDirectory() as tmp:
    for n in (6,3):
        subprocess.run([sys.executable,'deepseek_rounds.py','--mock','--max-rounds',str(n),'--output',str(Path(tmp)/f'mock{n}.json')],check=True)


In [ ]:
from getpass import getpass
try:
    from google.colab import userdata
    key=userdata.get('DEEPSEEK_API_KEY')
except Exception:
    key=getpass('Enter your DeepSeek API key (hidden input): ')
if not key:raise ValueError('The API key is empty')
os.environ['DEEPSEEK_API_KEY']=key
del key
print('API key set; it will not be written to results.')


## Live Experiment Settings
For the first run, use STAGE="pilot": 3 tasks per difficulty, 9 in total; one run per condition.
For the formal experiment, use STAGE="formal": 30 tasks per difficulty, 90 in total; three runs per condition, matching the previous experiment's scale.
Set FORMAL_PER_DIFFICULTY to 10 for 30 total tasks with three runs each. Do not confuse tasks per difficulty with the total number of tasks.
Each execution creates a new timestamped directory to avoid overwriting results. Repeated execution makes new billable API calls. On an API error, completed tasks are saved and execution stops; there is no automatic retry or resume.


In [ ]:
STAGE='pilot' # Start with a pilot; change to formal for the formal experiment
FORMAL_PER_DIFFICULTY=30 # 30 x 3 difficulty levels = 90 tasks; set to 10 for 30 total tasks
assert STAGE in ('pilot','formal')
PER_DIFFICULTY=3 if STAGE=='pilot' else FORMAL_PER_DIFFICULTY
RUNS=1 if STAGE=='pilot' else 3
from datetime import datetime
folder=ROOT/'outputs'/(STAGE+'_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
folder.mkdir(parents=True,exist_ok=False)
print('Tasks per condition',3*PER_DIFFICULTY,'Runs',RUNS,'Total request cap',3*PER_DIFFICULTY*RUNS*(6+3))
for mode,n in (('baseline6',6),('budget3',3)):
    os.environ['PILOT_MAX_CALLS']=str(3*PER_DIFFICULTY*RUNS*n)
    subprocess.run([sys.executable,'deepseek_rounds.py','--per-difficulty',str(PER_DIFFICULTY),'--runs',str(RUNS),
        '--max-rounds',str(n),'--output',str(folder/(mode+'.json'))],check=True)
print('Completed. Results directory: ',folder)


In [ ]:
import statistics,csv
runs={m:json.loads((folder/(m+'.json')).read_text()) for m in ('baseline6','budget3')}
for m,d in runs.items():
    assert not d['mock'] and d['error'] is None
    assert len(d['results'])==3*PER_DIFFICULTY*RUNS
    assert len(d['api_records'])==d['usage']['calls']==sum(x['rounds'] for x in d['results'])
    assert all(x['rounds']<=d['config']['max_rounds'] and len(x['events'])==x['rounds'] for x in d['results'])
    assert all(x['reasoning_mode']=='no_reasoning' for x in d['results'])
    for k in ('input_tokens','output_tokens','cached_input_tokens'):
        assert d['usage'][k]==sum(x['api_usage'][k] for x in d['results'])
assert [(x['id'],x['run']) for x in runs['baseline6']['results']]==[(x['id'],x['run']) for x in runs['budget3']['results']]
rows=[]
for m,d in runs.items():
    for r in range(RUNS):
        for difficulty in ('all','easy','medium','hard'):
            xs=[x for x in d['results'] if x['run']==r and (difficulty=='all' or x['difficulty']==difficulty)]
            rows.append(dict(mode=m,run=r,difficulty=difficulty,n=len(xs),accuracy=statistics.mean(x['final_correct'] for x in xs),
                strict_accuracy=statistics.mean(x['strict_correct'] for x in xs),avg_rounds=statistics.mean(x['rounds'] for x in xs),
                avg_tools=statistics.mean(x['tool_calls'] for x in xs),exhausted=sum(x['budget_exhausted'] for x in xs),
                parse_failures=sum(x['parse_failures'] for x in xs),policy_rejections=sum(x['policy_rejections'] for x in xs),seconds=sum(x['seconds'] for x in xs)))
with (folder/'per_run.csv').open('w',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
lines=['# When2Tool: Six-Round versus Three-Round Budget','',
       '90-task or 30-task calculator subset using the DeepSeek API. A parameter modification experiment, not a new algorithm or a strict numerical reproduction of the original paper.',
       'Original paper https://arxiv.org/abs/2605.09252','',
       '|Condition|Accuracy mean ± sample SD|Completed and correct rate|Mean rounds|Budget exhaustions|API requests|Input tokens|Output tokens|Seconds|',
       '|---|---:|---:|---:|---:|---:|---:|---:|---:|']
for m,d in runs.items():
    rr=[x for x in rows if x['mode']==m and x['difficulty']=='all'];acc=[x['accuracy'] for x in rr]
    sd=f'{statistics.stdev(acc):.4f}' if RUNS>1 else 'NA';u=d['usage']
    lines.append(f"|{m}|{statistics.mean(acc):.4f} ± {sd}|{statistics.mean(x['strict_accuracy'] for x in rr):.4f}|{statistics.mean(x['avg_rounds'] for x in rr):.2f}|{sum(x['exhausted'] for x in rr)}|{u['calls']}|{u['input_tokens']}|{u['output_tokens']}|{d['seconds']:.1f}|")
# Same-prefix counterfactual replay costs zero additional API calls.
base=runs['baseline6']['results']
late=sum(x['late_rescue'] for x in base)
prefix_correct=sum(x['prefix3']['strict_correct'] for x in base)
full_correct=sum(x['strict_correct'] for x in base)
free_replay=dict(n=len(base),prefix3_correct=prefix_correct,full6_correct=full_correct,late_rescues=late,
                model_calls_after_round3=sum(max(0,x['rounds']-3) for x in base))
(folder/'same_prefix_analysis.json').write_text(json.dumps(free_replay,indent=2))
lines+=['',f'Same-trajectory truncated replay at no API cost: completed and correct within three rounds {prefix_correct}/{len(base)}; completed and correct within six rounds {full_correct}/{len(base)}; task executions rescued by additional rounds: {late}.',
        f'Actual baseline6 model requests in rounds 4–6: {free_replay["model_calls_after_round3"]}.',
        'Independent runs involve generation randomness, so the entire score difference cannot be attributed to the budget. Same-prefix analysis isolates the truncation effect on the observed trajectories.',
        'Accuracy uses the author\'s answer evaluation; the completed and correct rate additionally requires done=true. Original logs are retained after budget exhaustion, without fabricating answers.',
        'Sample SD reflects variation across runs on the same task set and is NA for a single run; actual usage is taken from API usage records.']
(folder/'report.md').write_text(chr(10).join(lines));print(chr(10).join(lines))
if STAGE=='pilot':
    ratio=FORMAL_PER_DIFFICULTY*3/(PER_DIFFICULTY*RUNS)
    print('Formal runtime in minutes, linearly extrapolated from the pilot: ',round(sum(d['seconds'] for d in runs.values())*ratio/60,1))


In [ ]:
import shutil
from google.colab import files
archive=shutil.make_archive('/content/when2tool_round_budget_'+folder.name,'zip',root_dir=folder)
files.download(archive)


## Usage and Limitations
Start with STAGE=pilot: 3 tasks each for easy, medium, and hard, one run per condition. Send the ZIP for inspection before switching to formal.
The formal default is 90 tasks with three runs per condition, totaling 540 task executions. With FORMAL_PER_DIFFICULTY=10, there are 30 tasks with three runs per condition, totaling 180 task executions.
Allow 1–5 minutes for the pilot and 15–30 minutes for the 90-task formal experiment. These are advance estimates without live API validation of the new setup; extrapolation from measured pilot runtime is more reliable.
The initial eight tests and mock executions are free; live execution incurs API charges. The request count is capped, and output is limited to 1024 tokens per request; there is no hard monetary cap. Failures are not automatically retried, and completed tasks are saved individually.
Only the round limit changes; prompts, tools, and parser remain unchanged. Both conditions disable internal thinking and use no_reasoning, with no training.
The compatibility limitations of the strict XML parser are retained and logged. Relaxing the parser during this budget experiment would introduce a second variable.
Existing results cannot be overwritten; this notebook always creates a new results directory. It does not resume interrupted runs; repeated execution incurs new charges.
